# B3 — mixed displacement-stress: local integration refinement near the groove (attempt 5)

**A genuinely different lever this time.** The four prior attempts (scale=500, scale=250, two-stage curriculum, relative constitutive residual) all tweaked the LOSS FORMULA and none improved on attempt 1's own direct-P pooled RMS (**824.9%**, corrected 2026-10-26 from a mislabeled "8.2%" -- see `PROJECT_STATUS.md`). A CPU root-cause investigation into attempt 4's own failure found the relative-residual fix crippled the GRADIENT for exactly the high-stress groove-region points it was meant to help -- consistent with the groove being an inherently hard-to-resolve, spatially-localized feature, not a loss-weighting problem.

**This attempt tries a different lever entirely: the INTEGRATION RESOLUTION near the groove, not the loss formula.** `train_B3.py`'s own `--local_refine_region` flag (refining the DEM energy-integration quadrature for elements touching the groove, keeping the operator's own mesh completely unchanged) was already proven on real GPU training for the energy-only baseline: it roughly **HALVED** that formulation's own region-stress error (62.2% -> 29.7%, 2026-09-30 -- see `PROJECT_STATUS.md`). That technique has never been tried on the mixed-stress formulation. `mixed_stress_loss_B3_locally_refined` (new, in `mixed_stress_B3.py`) extends it to BOTH residuals this formulation actually trains on (constitutive consistency, equilibrium) -- the energy-only baseline only ever needed to refine a single scalar energy density, never a per-node stress field that must also be interpolated and gradient'd.

**CPU-verified before any GPU time** (`test_mixed_stress_B3_local_refine.py`, 3 checks, all passing): (1) an empty region mask exactly reproduces the plain, non-refined loss (no contamination); (2) a non-empty region refined at the SAME degree as the standard 2-point rule still exactly reproduces the plain loss (confirms the split-and-recombine is correct, not just a no-op); (3) a genuinely finer region quadrature (n_sub=6) changes the loss value and stays finite/differentiable through both network outputs. A quick CPU smoke test at full production resolution (6,840 elements) confirmed the same 4/6,840 region elements the energy-only baseline found, and a few real training iterations run without error.

**Config: attempt 1's own base** (`p_output_scale=500`, `w_equilibrium=28000`, absolute constitutive, no curriculum -- the configuration that produced the project's best-available direct-P checkpoint so far) **plus `--local_refine_region 1 --local_refine_n_sub 10`** (10 matches the level the energy-only baseline's own convergence sweep found already stable), isolating this ONE change from everything tried before. 50,000 iterations, same budget as every other B3 run.

**What to watch**: does `region_avg_sigma_xx_direct` pooled RMS improve from 824.9%, the way the baseline's own region-stress error roughly halved under the same technique? Or does the groove's difficulty for this formulation turn out to be a different, deeper issue (the gradient-collapse-adjacent mechanism already found) that quadrature resolution alone cannot fix? Either outcome is a real, reportable finding -- the first genuinely untried lever for item 1.

In [ ]:
# =====================================================================
#  CELL -- B3 mixed displacement-stress: attempt 5, LOCAL INTEGRATION
#  REFINEMENT near the groove (--local_refine_region 1), on top of
#  attempt 1's own base config (scale=500, w_equilibrium=28000,
#  absolute constitutive, no curriculum). See the markdown cell above
#  for the full rationale and CPU verification.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 -- if not, '
      'use Runtime > Restart session before trusting this run)')

print(f'\n{"=" * 90}')
print('=== Re-running the CPU local-refine correctness checks in this environment ===')
from omar_pfem.test_mixed_stress_B3_local_refine import (
    test_empty_region_matches_plain_loss_exactly,
    test_nonempty_region_at_matching_degree_matches_plain_loss,
    test_finer_region_quadrature_changes_the_loss_and_stays_differentiable,
)
test_empty_region_matches_plain_loss_exactly()
test_nonempty_region_at_matching_degree_matches_plain_loss()
test_finer_region_quadrature_changes_the_loss_and_stays_differentiable()

from omar_pfem.train_B3_mixed_stress import get_args, train
from omar_pfem.data.data_generate_B3_dataset import DEFAULT_RESOLUTION

device = torch.device('cuda')

R = '/content/drive/MyDrive/pfem_run'
OUTPUT_DIR = f'{R}/b3_mixed_stress_local_refine'
os.makedirs(OUTPUT_DIR, exist_ok=True)

args = get_args([
    '--n_iters', '50000',
    '--pretrain_iters', '0',
    '--p_output_scale', '500.0',
    '--w_constitutive', '1.0',
    '--w_equilibrium', '28000.0',
    '--local_refine_region', '1',
    '--local_refine_n_sub', '10',
    '--batch_size', '8',
    '--log_every', '500',
    '--ckpt_every', '5000',
    '--output_dir', OUTPUT_DIR,
    # relative_constitutive=0 (default, absolute residual) -- deliberately
    # attempt 1's own base, not attempt 4's -- isolating the local-refine
    # lever from the (already-tried-and-falsified) relative-residual lever.
])
n_elem = (DEFAULT_RESOLUTION[0] - 1) * (DEFAULT_RESOLUTION[1] - 1) * (DEFAULT_RESOLUTION[2] - 1)
print(f'\nTraining at {DEFAULT_RESOLUTION} -> {n_elem} elements, {args.n_iters} iterations, '
      f'batch_size={args.batch_size}, w_constitutive={args.w_constitutive}, '
      f'w_equilibrium={args.w_equilibrium}, p_output_scale={args.p_output_scale}, '
      f'local_refine_region={bool(args.local_refine_region)}, '
      f'local_refine_n_sub={args.local_refine_n_sub}, checkpoints -> {OUTPUT_DIR}')
print('Watch the per-iteration wall-clock rate first (as this project always does before '
      'trusting a long run) -- the energy-only baseline\'s own local-refine run matched its '
      'non-refined rate (~0.15s/iteration) only AFTER a real ~12x-slowdown bug was found and '
      'fixed (recomputing shape functions for the full 6,840-element mesh every iteration '
      'instead of just the ~4 region elements); this run\'s own precompute is already built '
      'the fixed way, but confirming the real rate on GPU is still worth watching once before '
      'committing the full 50,000 iterations.')

torch.cuda.reset_peak_memory_stats(device)
t0 = time.time()
model = train(args, device)
elapsed_total = time.time() - t0

print(f'\n{"=" * 90}')
print(f'Training finished in {elapsed_total:.1f}s ({elapsed_total / args.n_iters:.4f}s/iteration average).')
print(f'GPU peak memory during training: '
      f'{torch.cuda.max_memory_allocated(device) / 1e6:.1f}MB allocated, '
      f'{torch.cuda.max_memory_reserved(device) / 1e6:.1f}MB reserved.')

# =====================================================================
#  Evaluation: same held-out FEM dataset every other B3 checkpoint uses,
#  now also including the H1 semi-norm metric (added 2026-10-26).
# =====================================================================
print(f'\n{"=" * 90}')
print('=== Evaluating the final checkpoint against the held-out FEM dataset ===')
CHECKPOINT = f'{OUTPUT_DIR}/checkpoint_{args.n_iters}.pt'
DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
OUT_JSON = f'{OUTPUT_DIR}/qois_{args.n_iters}.json'

sys.argv = [
    'evaluate_B3_mixed_stress.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--out_json', OUT_JSON,
]
from omar_pfem.evaluate_B3_mixed_stress import main as evaluate_main
evaluate_main()

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(OUTPUT_DIR, kind='b3_mixed_stress_local_refine_training',
                    args={'n_iters': args.n_iters, 'pretrain_iters': args.pretrain_iters,
                          'batch_size': args.batch_size,
                          'resolution': DEFAULT_RESOLUTION, 'lr': args.lr,
                          'w_constitutive': args.w_constitutive,
                          'w_equilibrium': args.w_equilibrium,
                          'p_output_scale': args.p_output_scale,
                          'local_refine_region': bool(args.local_refine_region),
                          'local_refine_n_sub': args.local_refine_n_sub},
                    started_at=_started,
                    results={'elapsed_total_s': elapsed_total,
                             's_per_iter': elapsed_total / args.n_iters},
                    outputs=[CHECKPOINT, OUT_JSON],
                    notes="Item 1 attempt 5 (2026-10-26): local integration refinement near "
                          "the groove for the constitutive+equilibrium residuals, reusing the "
                          "technique train_B3.py's own --local_refine_region already proved on "
                          "real GPU training for the energy-only baseline (roughly halved its "
                          "own region-stress error, 62.2%->29.7%, 2026-09-30). Built on "
                          "attempt 1's own base config (scale=500, w_equilibrium=28000, "
                          "absolute constitutive) -- the only change is "
                          "local_refine_region=1 -- isolating this one, genuinely different "
                          "lever (integration resolution, not loss formula) from the four "
                          "prior attempts.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back, especially '
      '"region_avg_sigma_xx_direct" pooled RMS -- compare against attempt 1\'s own 824.9%.')
